In [1]:
from torch_geometric.nn import MessagePassing

/Users/aman/opt/anaconda3/envs/fraud/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
class myGAT(MessagePassing):

    def __init__(self, in_channels, out_channels, heads = 2,
                 negative_slope = 0.2, dropout = 0., **kwargs):
        super(myGAT, self).__init__(node_dim=0, **kwargs)

        self.in_channels = in_channels 
        self.out_channels = out_channels 
        self.heads = heads 
        self.negative_slope = negative_slope
        self.dropout = dropout

        self.lin_l = None
        self.lin_r = None
        self.att_l = None
        self.att_r = None
     
        
        self.lin_l = Linear(in_channels, heads*out_channels)
        self.lin_r = self.lin_l

        self.att_l = Parameter(torch.Tensor(1, heads, out_channels).float())
        self.att_r = Parameter(torch.Tensor(1, heads, out_channels).float())

        self.reset_parameters()

    def reset_parameters(self):
        nn.init.xavier_uniform_(self.lin_l.weight)
        nn.init.xavier_uniform_(self.lin_r.weight)
        nn.init.xavier_uniform_(self.att_l)
        nn.init.xavier_uniform_(self.att_r)

    def forward(self, x, edge_index, size = None):
        
        H, C = self.heads, self.out_channels 

        
        x_source = self.lin_l(x).view(-1,H,C) 
        x_target = self.lin_r(x).view(-1,H,C) 

        
        alpha_l = (x_source * self.att_l).sum(dim=-1) 
        alpha_r = (x_target * self.att_r).sum(dim=-1) 

        
        out = self.propagate(edge_index, x=(x_source, x_target), alpha=(alpha_l, alpha_r),size=size) 
        out = out.view(-1, self.heads * self.out_channels) 

        return out

    def message(self, x_j, alpha_j, alpha_i, index, ptr, size_i):
        
        attention = F.leaky_relu((alpha_j + alpha_i), self.negative_slope) 
        attention = softmax(attention, index, ptr, size_i) 
        attention = F.dropout(attention, p=self.dropout, training=self.training) 

        
        out = x_j * attention.unsqueeze(-1)  

        return out

    def aggregate(self, inputs, index, dim_size = None):
      
        out = torch_scatter.scatter(inputs, index, dim=self.node_dim, 
                                    dim_size=dim_size, reduce='sum')
  
        return out